# Video Maker Quick — Multi-Scene CZI
Full-width (no ROI crop) video export for every scene in a multi-scene CZI file.
Includes an interactive preview of channel superposition before rendering.

## 1 — Imports

In [1]:
from chipanalysis.utils.file_reader import get_frame, get_pixel_sizes_um, get_timestamps_by_T, stretch_contrast
from chipanalysis.utils.maye_video_axio import norm, make_annotated, mcherry, gfp, gray_cmap, clamp

from pathlib import Path
from aicspylibczi import CziFile
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from moviepy import ImageClip, concatenate_videoclips
from statistics import median

%load_ext autoreload
%autoreload 2
%matplotlib widget

## 2 — Load CZI file and inspect scenes

In [ ]:
root_path = "/Users/bisot/Documents/PostDoc2/test_data/ICP"
czi_name  = "ICP15-03.czi"

czi_path   = Path(root_path, czi_name)
czi        = CziFile(czi_path)
pixel_size = get_pixel_sizes_um(czi)

dims      = czi.dims
sizes     = czi.size
dim_sizes = dict(zip(dims, sizes))

n_scenes   = dim_sizes.get("S", 1)
n_times    = dim_sizes.get("T", 1)
n_channels = dim_sizes.get("C", 1)

print(f"Dims      : {dims}")
print(f"Dim sizes : {dim_sizes}")
print(f"Scenes    : {n_scenes}  (indices 0 – {n_scenes-1})")
print(f"Timepoints: {n_times}")
print(f"Channels  : {n_channels}")
print(f"Pixel size: {pixel_size['X']:.4f} µm")

# Scene bounding boxes
scene_bboxes = czi.get_all_scene_bounding_boxes()
for s, bb in scene_bboxes.items():
    print(f"  scene {s}: x={bb.x}, y={bb.y}, w={bb.w}, h={bb.h}")

# Timestamps (use C=0 as reference)
times = get_timestamps_by_T(czi, C=0, Z=0)

.//Scaling//Distance[@Id='X']//Value
.//Scaling//Distance[@Id='Y']//Value
Dims      : STCMYX
Dim sizes : {'S': 8, 'T': 386, 'C': 3, 'M': 4, 'Y': 896, 'X': 896}
Scenes    : 8  (indices 0 – 7)
Timepoints: 386
Channels  : 3
Pixel size: 2.7438 µm
  scene 0: x=24083, y=11742, w=3316, h=896
  scene 1: x=24073, y=14245, w=3315, h=896
  scene 2: x=24096, y=16757, w=3315, h=896
  scene 3: x=24120, y=19283, w=3315, h=896
  scene 4: x=35367, y=12483, w=3315, h=896
  scene 5: x=35375, y=15029, w=3315, h=896
  scene 6: x=35398, y=17513, w=3315, h=896
  scene 7: x=35444, y=20064, w=3315, h=896


## 3 — Channel colour mapping & contrast settings

Edit the dict below to control which colormap and contrast percentiles each channel uses.

In [4]:
# ── channel config ──────────────────────────────────────────────
# channel index → { cmap, gamma, stretch_min%, stretch_max% }
# lo/hi are set automatically per-scene below; override manually if needed.
CHANNEL_CFG = {
    0: dict(cmap=gfp,       gamma=0.45, stretch_min=90, stretch_max=99.5),  # green (GFP)
    1: dict(cmap=mcherry,   gamma=0.45, stretch_min=90, stretch_max=99.5),  # magenta (mCherry)
    2: dict(cmap=gray_cmap, gamma=1.0,  stretch_min=1,  stretch_max=99),    # brightfield / gray
}

def build_merged(frames_dict):
    """frames_dict: {ch_idx: 2-D float32 array already contrast-stretched & normed}"""
    merged = np.zeros((*next(iter(frames_dict.values())).shape, 3), dtype=np.float32)
    for ch, img in frames_dict.items():
        rgb = CHANNEL_CFG[ch]["cmap"](img)[..., :3]
        merged += rgb
    return np.clip(merged, 0, 1)

## 4 — Interactive preview: pick a scene & frame, visualise channel superposition

In [5]:
scene_slider = widgets.IntSlider(value=0, min=0, max=n_scenes-1, description="Scene:")
time_slider  = widgets.IntSlider(value=0, min=0, max=n_times-1,  description="Frame:")

out = widgets.Output()

def _preview(scene_idx, time_idx):
    out.clear_output(wait=True)
    frames = {}
    for ch, cfg in CHANNEL_CFG.items():
        raw, _ = get_frame(czi, time_idx, ch, scene=scene_idx)
        stretched = stretch_contrast(raw, cfg["stretch_min"], cfg["stretch_max"])
        if cfg["gamma"] != 1.0:
            stretched = np.clip(stretched, 0, 1) ** cfg["gamma"]
        frames[ch] = norm(stretched)

    merged = build_merged(frames)

    with out:
        fig, axes = plt.subplots(1, len(CHANNEL_CFG) + 1,
                                  figsize=(4 * (len(CHANNEL_CFG) + 1), 4))
        cmaps_list = [CHANNEL_CFG[ch]["cmap"] for ch in sorted(CHANNEL_CFG)]
        for ax, (ch, img), cmap in zip(axes, sorted(frames.items()), cmaps_list):
            ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(f"Ch {ch}", fontsize=10)
            ax.axis("off")
        axes[-1].imshow(merged)
        axes[-1].set_title("Merged", fontsize=10)
        axes[-1].axis("off")
        fig.suptitle(f"Scene {scene_idx} — frame {time_idx}", fontsize=12)
        plt.tight_layout()
        plt.show()

interactive = widgets.interactive(_preview, scene_idx=scene_slider, time_idx=time_slider)
display(widgets.VBox([widgets.HBox([scene_slider, time_slider]), out]))
_preview(0, 0)  # initial render

## 5 — Set contrast limits (auto from frame 0 of each scene)

Run this after you are happy with the preview scene/frame selection.
`lo`/`hi` values are computed once per scene and reused during video rendering.

In [6]:
# scene_contrast[scene_idx][ch] = (lo, hi)
scene_contrast = {}

for s in range(n_scenes):
    scene_contrast[s] = {}
    for ch, cfg in CHANNEL_CFG.items():
        raw, _ = get_frame(czi, 0, ch, scene=s)
        lo, hi = np.percentile(raw, (cfg["stretch_min"], cfg["stretch_max"]))
        scene_contrast[s][ch] = (lo, hi)
        print(f"  scene {s} ch {ch}: lo={lo:.1f}  hi={hi:.1f}")

print("✅ Contrast limits computed.")

  scene 0 ch 0: lo=4157.0  hi=7040.0
  scene 0 ch 1: lo=169.0  hi=1063.0
  scene 0 ch 2: lo=2056.0  hi=12662.0
  scene 1 ch 0: lo=4698.0  hi=7770.0
  scene 1 ch 1: lo=169.0  hi=1133.0
  scene 1 ch 2: lo=1764.0  hi=11200.0
  scene 2 ch 0: lo=6965.0  hi=10314.0
  scene 2 ch 1: lo=160.0  hi=573.0
  scene 2 ch 2: lo=1807.0  hi=11373.0
  scene 3 ch 0: lo=8998.0  hi=13912.0
  scene 3 ch 1: lo=164.0  hi=888.0
  scene 3 ch 2: lo=1612.0  hi=11450.0
  scene 4 ch 0: lo=2723.0  hi=4875.0
  scene 4 ch 1: lo=202.0  hi=1407.0
  scene 4 ch 2: lo=2041.0  hi=12616.0
  scene 5 ch 0: lo=2613.0  hi=4813.0
  scene 5 ch 1: lo=168.0  hi=1116.0
  scene 5 ch 2: lo=2109.0  hi=12024.0
  scene 6 ch 0: lo=3783.0  hi=6805.0
  scene 6 ch 1: lo=173.0  hi=1111.0
  scene 6 ch 2: lo=1739.0  hi=11542.0
  scene 7 ch 0: lo=5498.0  hi=7937.0
  scene 7 ch 1: lo=167.0  hi=815.0
  scene 7 ch 2: lo=1670.0  hi=11159.0
✅ Contrast limits computed.


## 6 — Video rendering function (single scene, full width)

In [7]:
def make_scene_video(
    czi,
    scene_idx,
    times,
    pixel_size_x,
    output_path,
    contrast=None,      # dict {ch: (lo, hi)}, auto-computed if None
    scale_factor=1800.0,
    fps=10,
    resize_width=1024,
):
    """Render a time-lapse video for one scene (full image width, all channels merged)."""

    if contrast is None:
        contrast = {}
        for ch, cfg in CHANNEL_CFG.items():
            raw, _ = get_frame(czi, 0, ch, scene=scene_idx)
            lo, hi = np.percentile(raw, (cfg["stretch_min"], cfg["stretch_max"]))
            contrast[ch] = (lo, hi)

    # Build per-frame durations from timestamps
    ts = [s for _, s in times]
    real_deltas = [None]
    for k in range(1, len(ts)):
        dt = max((ts[k] - ts[k-1]).total_seconds(), 0.0)
        real_deltas.append(dt)
    positive = [d for d in real_deltas[1:] if d and d > 0]
    baseline = median(positive) if positive else 1.0
    real_deltas[0] = baseline
    if len(real_deltas) > 1:
        real_deltas[-1] = real_deltas[-2] if real_deltas[-2] is not None else baseline
    video_durations = [clamp(float(d) / scale_factor, 0.0, 1e12) for d in real_deltas]

    clips = []
    for (time_i, _), dur in zip(times, video_durations):
        frames = {}
        for ch, cfg in CHANNEL_CFG.items():
            lo, hi = contrast[ch]
            raw, _ = get_frame(czi, time_i, ch, scene=scene_idx, lo=lo, hi=hi)
            stretched = stretch_contrast(raw, lo=lo, hi=hi)
            if cfg["gamma"] != 1.0:
                stretched = np.clip(stretched, 0, 1) ** cfg["gamma"]
            frames[ch] = norm(stretched)

        merged = build_merged(frames)

        annotated = make_annotated(
            merged,
            time_i,
            times,
            pixel_size_x,
            resize_width=resize_width,
            mode="RGB",
        )

        if annotated.dtype != np.uint8:
            frame_u8 = (np.clip(annotated, 0, 1) * 255).astype(np.uint8) \
                       if annotated.max() <= 1.0 else annotated.astype(np.uint8)
        else:
            frame_u8 = annotated

        clips.append(ImageClip(frame_u8, duration=dur))

    final = concatenate_videoclips(clips, method="compose")
    final.write_videofile(
        str(output_path),
        fps=fps,
        codec="libx264",
        audio=False,
        ffmpeg_params=[
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            "-profile:v", "baseline",
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
        ],
    )
    print(f"✅ Scene {scene_idx} → {output_path}")

## 7 — Generate videos for all scenes

Set `output_dir` and `scenes_to_render` (use `None` for all scenes), then run.

In [8]:
root_path

'/Users/bisot/Documents/PostDoc2/test_data'

In [9]:
output_dir     = Path(root_path)
scenes_to_render = None   # None = all scenes, or e.g. [0, 2, 5]
scale_factor   = 1800.0   # time-lapse acceleration factor
fps            = 10
resize_width   = 1024

scene_list = list(range(n_scenes)) if scenes_to_render is None else scenes_to_render
stem = Path(czi_name).stem

for s in scene_list:
    out_path = output_dir / f"{stem}_scene{s:02d}.mp4"
    print(f"\n── Rendering scene {s}/{n_scenes-1} → {out_path.name}")
    make_scene_video(
        czi,
        scene_idx=s,
        times=times,
        pixel_size_x=pixel_size["X"],
        output_path=out_path,
        contrast=scene_contrast.get(s),   # uses pre-computed limits if available
        scale_factor=scale_factor,
        fps=fps,
        resize_width=resize_width,
    )

print("\n🎬 All done!")


── Rendering scene 0/7 → ICP16-03_scene00.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene00.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene00.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene00.mp4
✅ Scene 0 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene00.mp4

── Rendering scene 1/7 → ICP16-03_scene01.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene01.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene01.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene01.mp4
✅ Scene 1 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene01.mp4

── Rendering scene 2/7 → ICP16-03_scene02.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene02.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene02.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene02.mp4
✅ Scene 2 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene02.mp4

── Rendering scene 3/7 → ICP16-03_scene03.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene03.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene03.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene03.mp4
✅ Scene 3 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene03.mp4

── Rendering scene 4/7 → ICP16-03_scene04.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene04.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene04.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene04.mp4
✅ Scene 4 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene04.mp4

── Rendering scene 5/7 → ICP16-03_scene05.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene05.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene05.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene05.mp4
✅ Scene 5 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene05.mp4

── Rendering scene 6/7 → ICP16-03_scene06.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene06.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene06.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene06.mp4
✅ Scene 6 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene06.mp4

── Rendering scene 7/7 → ICP16-03_scene07.mp4
MoviePy - Building video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene07.mp4.
MoviePy - Writing video /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene07.mp4



MoviePy - Done !
MoviePy - video ready /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene07.mp4
✅ Scene 7 → /Users/bisot/Documents/PostDoc2/test_data/ICP16-03_scene07.mp4

🎬 All done!
